In [9]:
import pandas as pd
import matplotlib.pyplot as plt
# CSV path + output file
CSV_PATH = "C:/Users/14794/Desktop/Longterm memory/us_H_paths_CI_field_bootstrap_with_sums_90.csv"
OUT_PNG  = "USA_H_pairwise_sums_CI_6PANELS_90.png"

# US subperiods
PERIODS = [
    ("Period 1", "2016-12-05", "2017-08-01"),
    ("Period 2", "2017-08-02", "2018-09-01"),
    ("Period 3", "2018-09-02", "2020-02-01"),
    ("Period 4", "2020-02-02", "2023-09-01"),
    ("Period 5", "2023-09-02", "2025-12-31"),
]

# ============================================================
# Plotting functions
def add_subperiod_separators(ax, periods, *, color="red", linestyle=":", linewidth=1.2, alpha=0.9):
    # draw vertical dotted lines at the START of each subperiod
    for _, start, _ in periods[1:]:
        ax.axvline(pd.to_datetime(start), color=color, linestyle=linestyle,
                   linewidth=linewidth, alpha=alpha, zorder=10)

def plot_six_sum_panels_us(ci_df: pd.DataFrame, periods, out_png: str, *, y_label="H sum"):
    # titles
    title_map = {
        "S_H11": "H(Stock) + H(Stock)",
        "S_H12": "H(Stock) + H(Volatility)",
        "S_H13": "H(Stock) + H(Interest Rate)",
        "S_H22": "H(Volatility) + H(Volatility)",
        "S_H23": "H(Volatility) + H(Interest Rate)",
        "S_H33": "H(Interest Rate) + H(Interest Rate)",
    }

    panels = [
        ("S_H11", "S_H11_lo", "S_H11_hi"),
        ("S_H12", "S_H12_lo", "S_H12_hi"),
        ("S_H13", "S_H13_lo", "S_H13_hi"),
        ("S_H22", "S_H22_lo", "S_H22_hi"),
        ("S_H23", "S_H23_lo", "S_H23_hi"),
        ("S_H33", "S_H33_lo", "S_H33_hi"),
    ]

    fig, axes = plt.subplots(3, 2, figsize=(14, 12), sharex=True)
    axes = axes.ravel()

    for ax, (mid, lo, hi) in zip(axes, panels):
        # CI band first (line stays crisp on top)
        ax.fill_between(ci_df["date"], ci_df[lo], ci_df[hi], alpha=0.25, zorder=1)
        ax.plot(ci_df["date"], ci_df[mid], linewidth=1.5, zorder=3)

        ax.axhline(1.0, color="mediumseagreen", linestyle=":", linewidth=3.0, alpha=0.9, zorder=2)

        add_subperiod_separators(ax, periods)

        ax.set_title(title_map.get(mid, mid))
        ax.set_ylabel(y_label)
        ax.grid(True, alpha=0.25)

    # x-label only bottom row
    axes[-1].set_xlabel("Date")
    axes[-2].set_xlabel("Date")

    plt.tight_layout()
    fig.savefig(out_png, dpi=200)
    plt.close(fig)

# ============================================================
# Load CSV and plot
df = pd.read_csv(CSV_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

plot_six_sum_panels_us(df, PERIODS, OUT_PNG)
print("Saved:", OUT_PNG)

Saved: CN_H_pairwise_sums_CI_6PANELS_90.png
